# **Samsung Canada - Multi-Category Web Scraper**

This notebook automates web scraping across **all major product categories** on Samsung Canada (`samsung.com/ca`).

### **Scraping Strategy:**
- **Catalog-Only Extraction**: Scrapes all information directly from the category catalog page (`/ca/<category>/all-<category>/`) without loading individual product detail pages.
- **Full Variant Matrix**: Interactively loops through all **Color Options** and all **Storage Options for each Color** directly on each product card to capture variant-level pricing, financing, and stock.
- **Exact Review Counts**: Extracts exact counts even when display shows `>9,999`.
- **Clean Separation of Badges & Offers**: Distinguishes between device tags/badges (`Badge`: e.g. `New`, `Online Exclusive`) and promotional discounts (`Offers`: e.g. `Save up to $490 with an eligible trade-in`, coupon codes, instant savings).

### **Categories Covered:**
1. **Smartphones** (`/ca/smartphones/all-smartphones/`)
2. **Tablets** (`/ca/tablets/all-tablets/`)
3. **Watches / Smartwatches** (`/ca/watches/all-watches/`)
4. **Audio & Galaxy Buds** (`/ca/audio-sound/all-audio-sound/`)
5. **TVs** (`/ca/tvs/all-tvs/`)
6. **Soundbars & Home Audio** (`/ca/audio-devices/all-audio-devices/`)
7. **Refrigerators** (`/ca/refrigerators/all-refrigerators/`)
8. **Laundry (Washers & Dryers)** (`/ca/laundry/all-laundry/`)
9. **Cooking Appliances** (`/ca/cooking-appliances/all-cooking-appliances/`)
10. **Monitors** (`/ca/monitors/all-monitors/`)
11. **Computers & Galaxy Books** (`/ca/computers/all-computers/`)

---

### **Output Organization:**
Each category is saved into its **own separate folder** as a distinct `.csv` file:
```
scraped_data/
 |-- Smartphones/
 |   `-- samsung_smartphones.csv
 |-- Tablets/
 |   `-- samsung_tablets.csv
 |-- Watches/
 |   `-- samsung_watches.csv
 |-- Audio_and_Buds/
 |   `-- samsung_audio_and_buds.csv
 |-- TVs/
 |   `-- samsung_tvs.csv
 |-- Sound_Devices/
 |   `-- samsung_sound_devices.csv
 |-- Refrigerators/
 |   `-- samsung_refrigerators.csv
 |-- Laundry/
 |   `-- samsung_laundry.csv
 |-- Cooking_Appliances/
 |   `-- samsung_cooking_appliances.csv
 |-- Monitors/
 |   `-- samsung_monitors.csv
 `-- Computers/
     `-- samsung_computers.csv
```

### **Data Fields Extracted:**
- `Model Name`: Base device model name
- `Color`: Selected color variant
- `Storage`: Selected capacity / size variant
- `SKU Code`: Exact variant model code
- `Product Price`: Current selling price
- `Original Price`: Was/MSRP regular price
- `Financing Option`: Monthly installment financing text
- `Badge`: Tag or badge (e.g. `New`, `Online Exclusive`)
- `Offers`: Promotional offers (e.g. `Save up to $490 with an eligible trade-in`, coupon codes, savings)
- `Product Rating`: Star rating score
- `Number of Ratings`: Exact review count (>9,999 extracted from title/tooltip)
- `Stock Status`: `In Stock` vs `Notify Me` / `Out of Stock`
- `Product URL`: Variant URL

## **1. Install Required Libraries (If Not Already Installed)**

In [46]:
# Install any dependencies
# !pip install selenium webdriver-manager beautifulsoup4 pandas

## **2. Import Necessary Libraries**

In [47]:
import os
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("Libraries imported successfully!")

Libraries imported successfully!


## **3. Configure Selenium WebDriver**

The function below creates a Chrome WebDriver with anti-bot headers and optimized settings. 
- Set `headless=False` if you want to watch the browser interact with cards live.
- Set `headless=True` for fast, silent execution in the background.

In [48]:
def init_webdriver(headless=False):
    """Initializes and returns a configured Chrome WebDriver."""
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # 1. First try Selenium 4's built-in driver management
    try:
        return webdriver.Chrome(options=opts)
    except Exception:
        pass
        
    # 2. Fallback to webdriver-manager with Windows path correction
    try:
        driver_path = ChromeDriverManager().install()
        if not driver_path.endswith(".exe"):
            parent = os.path.dirname(driver_path)
            candidate = os.path.join(parent, "chromedriver.exe")
            if os.path.exists(candidate):
                driver_path = candidate
        return webdriver.Chrome(service=Service(driver_path), options=opts)
    except Exception:
        return webdriver.Chrome(options=opts)

## **4. Define Product Categories and Output Paths**

Each category is mapped to its landing URL on Samsung Canada, its target directory, and output `.csv` filename.

In [49]:
BASE_OUTPUT_DIR = "scraped_data"

CATEGORIES = {
    "Smartphones": {
        "url": "https://www.samsung.com/ca/smartphones/all-smartphones/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Smartphones"),
        "filename": "samsung_smartphones.csv"
    },
    "Tablets": {
        "url": "https://www.samsung.com/ca/tablets/all-tablets/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Tablets"),
        "filename": "samsung_tablets.csv"
    },
    "Watches": {
        "url": "https://www.samsung.com/ca/watches/all-watches/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Watches"),
        "filename": "samsung_watches.csv"
    },
    "Audio_and_Buds": {
        "url": "https://www.samsung.com/ca/audio-sound/all-audio-sound/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Audio_and_Buds"),
        "filename": "samsung_audio_and_buds.csv"
    },
    "TVs": {
        "url": "https://www.samsung.com/ca/tvs/all-tvs/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "TVs"),
        "filename": "samsung_tvs.csv"
    },
    "Sound_Devices": {
        "url": "https://www.samsung.com/ca/audio-devices/all-audio-devices/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Sound_Devices"),
        "filename": "samsung_sound_devices.csv"
    },
    "Refrigerators": {
        "url": "https://www.samsung.com/ca/refrigerators/all-refrigerators/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Refrigerators"),
        "filename": "samsung_refrigerators.csv"
    },
    "Laundry": {
        "url": "https://www.samsung.com/ca/laundry/all-laundry/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Laundry"),
        "filename": "samsung_laundry.csv"
    },
    "Cooking_Appliances": {
        "url": "https://www.samsung.com/ca/cooking-appliances/all-cooking-appliances/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Cooking_Appliances"),
        "filename": "samsung_cooking_appliances.csv"
    },
    "Monitors": {
        "url": "https://www.samsung.com/ca/monitors/all-monitors/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Monitors"),
        "filename": "samsung_monitors.csv"
    },
    "Computers": {
        "url": "https://www.samsung.com/ca/computers/all-computers/",
        "folder": os.path.join(BASE_OUTPUT_DIR, "Computers"),
        "filename": "samsung_computers.csv"
    }
}

print(f"Configured {len(CATEGORIES)} categories:")
for name in CATEGORIES:
    print(f" - {name}")

Configured 11 categories:
 - Smartphones
 - Tablets
 - Watches
 - Audio_and_Buds
 - TVs
 - Sound_Devices
 - Refrigerators
 - Laundry
 - Cooking_Appliances
 - Monitors
 - Computers


## **5. Core Catalog-Only Scraping & Variant Parsing Logic**

These functions handle:
1. `handle_cookie_consent()`: Automatically accepts cookie consent banners.
2. `scroll_and_load_all_products()`: Scrolls dynamically and clicks 'View More' to reveal all catalog cards.
3. `scrape_catalog_page_variants()`: Iterates over each card, clicks through all **Color** and **Storage** swatches, and extracts variant pricing, financing, badges, promotional offers, exact ratings, and stock status.

In [50]:
def handle_cookie_consent(driver):
    """Dismisses cookie consent banners if present."""
    try:
        banner = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.ID, "truste-consent-track"))
        )
        btn = banner.find_element(By.XPATH, ".//button[contains(text(), 'Accept') or text()='Accept All']")
        btn.click()
        time.sleep(1)
    except Exception:
        pass
    
    try:
        onetrust_btn = driver.find_elements(By.ID, "onetrust-accept-btn-handler")
        if onetrust_btn:
            onetrust_btn[0].click()
            time.sleep(1)
    except Exception:
        pass

def click_view_more_if_available(driver):
    """Clicks 'View More' button if present to reveal additional products."""
    view_more_xpaths = [
        "//button[contains(@class, 'pd21-product-finder__view-more')]",
        "//button[contains(@class, 'js-pfv2-view-more')]",
        "//a[contains(@class, 'js-pfv2-view-more-cta')]",
        "//button[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'view more')]"
    ]
    for xpath in view_more_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xpath)
            for b in buttons:
                if b.is_displayed():
                    driver.execute_script("arguments[0].click();", b)
                    time.sleep(2)
        except Exception:
            continue

def scroll_and_load_all_products(driver, max_scrolls=15, scroll_delay=2.5):
    """Scrolls down the category page and clicks 'View More' until all products are loaded."""
    last_height = driver.execute_script("return document.body.scrollHeight")
    scroll_count = 0
    
    while scroll_count < max_scrolls:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_delay)
        click_view_more_if_available(driver)
        
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
        scroll_count += 1

def scrape_catalog_page_variants(driver, category_name, category_url, max_cards=None):
    """
    Scrapes products directly on the catalog page cards without loading product detail pages.
    Iterates card-by-card to avoid browser script timeouts and provide real-time progress.
    Extracts all Color and Storage variants, pricing, financing, badges, offers, and stock status.
    """
    print(f"\n[Catalog Scraper] Loading category: {category_name} -> {category_url}")
    driver.get(category_url)
    handle_cookie_consent(driver)
    scroll_and_load_all_products(driver)

    js_card_extractor = """
    var callback = arguments[arguments.length - 1];
    var cardIndex = arguments[0];

    (async function() {
        var allCards = Array.from(document.querySelectorAll('.pd21-product-card__item.js-pfv2-product-card, .pd21-product-card__item, .pd03-product-card'));
        if (cardIndex < 0 || cardIndex >= allCards.length) {
            callback([]);
            return;
        }

        var card = allCards[cardIndex];
        try {
            card.scrollIntoView({ behavior: 'instant', block: 'center' });
            await new Promise(function(r) { setTimeout(r, 200); });
        } catch(e) {}

        // 1. Model / Product Name
        var nameEl = card.querySelector('.pd21-product-card__name, [class*="product-card__name"], .product-name, h2, h3');
        var baseName = nameEl ? nameEl.innerText.trim() : '';
        if (!baseName) {
            await new Promise(function(r) { setTimeout(r, 300); });
            nameEl = card.querySelector('.pd21-product-card__name, [class*="product-card__name"], .product-name, h2, h3');
            baseName = nameEl ? nameEl.innerText.trim() : 'Unknown Product';
        }
        if (!baseName || baseName === 'Unknown Product') {
            callback([]);
            return;
        }

        var results = [];

        // 2. Rating & Exact Review Count
        var ratingEl = card.querySelector('.pd21-product-card__rating, [class*="rating"]');
        var ratingScore = null;
        var exactCount = 0;
        if (ratingEl) {
            var scoreEl = ratingEl.querySelector('strong span, .rating__point span:not(.hidden), strong');
            if (scoreEl) {
                var mScore = scoreEl.innerText.trim().match(/([0-5](?:\\.\\d+)?)/);
                if (mScore) ratingScore = parseFloat(mScore[1]);
            }

            // Check title attribute which contains exact count even for >9,999 reviews (e.g. title="4.8, (37185), Galaxy...")
            var titleText = ratingEl.getAttribute('title') || '';
            var mTitle = titleText.match(/\\(([0-9,]+)\\)/);
            if (mTitle) {
                exactCount = parseInt(mTitle[1].replace(/,/g, ''), 10);
            } else {
                var countEl = ratingEl.querySelector('em span:not(.hidden), em, [class*="review-count"]');
                if (countEl) {
                    var mCount = countEl.innerText.trim().match(/([0-9,]+)/);
                    if (mCount) exactCount = parseInt(mCount[1].replace(/,/g, ''), 10);
                }
            }
        }

        // 3. Badges / Tags (e.g. 'New', 'Online Exclusive')
        var badgeList = [];
        var badgeEls = card.querySelectorAll('.pd21-product-card__badge-wrap .badge-icon, .pd21-product-card__badge span, .badge-icon, .pd21-product-card__tag');
        badgeEls.forEach(function(b) {
            var bTxt = b.innerText.trim();
            if (bTxt && !badgeList.includes(bTxt)) badgeList.push(bTxt);
        });
        var badgeStr = badgeList.length > 0 ? badgeList.join(' | ') : 'None';

        // 4. Promotional Offers (e.g. Trade-in discounts, coupon codes, instant savings)
        var offersList = [];
        var tradeInEls = card.querySelectorAll('[class*="trade-in-message"], [class*="trade-in"], .seca-s26-fe-trade-in-message');
        tradeInEls.forEach(function(t) {
            var tTxt = t.innerText.trim().replace(/\\s+/g, ' ');
            if (tTxt && !offersList.includes(tTxt)) offersList.push(tTxt);
        });

        var couponEls = card.querySelectorAll('span[class*="-msg"], span[id*="COUPON"], [class*="promo-code"]');
        couponEls.forEach(function(c) {
            var cTxt = c.innerText.trim().replace(/\\s+/g, ' ');
            if (cTxt && !offersList.includes(cTxt)) offersList.push(cTxt);
        });

        var saveEl = card.querySelector('.price-ux__save, [class*="save"]');
        if (saveEl && saveEl.innerText.trim()) {
            var sTxt = saveEl.innerText.trim().replace(/\\s+/g, ' ');
            if (!offersList.includes(sTxt)) offersList.push(sTxt);
        }
        var origPriceEl = card.querySelector('.price-ux__price-original');
        if (origPriceEl) {
            var mSave = origPriceEl.innerText.match(/Save\\s+\\$[0-9,]+(?:\\.[0-9]{2})?/i);
            if (mSave && !offersList.includes(mSave[0])) {
                offersList.push(mSave[0]);
            }
        }

        var promoEl = card.querySelector('.pd21-product-card__promo, [class*="promo"]:not([class*="badge"])');
        if (promoEl && promoEl.innerText.trim()) {
            var pTxt = promoEl.innerText.trim().replace(/\\s+/g, ' ');
            if (!offersList.includes(pTxt)) offersList.push(pTxt);
        }
        var offersStr = offersList.length > 0 ? offersList.join(' | ') : 'None';

        // Helper to check if button is selected
        function isBtnSelected(btn) {
            if (!btn) return false;
            var blindText = btn.querySelector('.blind, .hidden')?.innerText || '';
            if (blindText.toLowerCase().includes('selected')) return true;
            if (btn.classList.contains('is-checked') || btn.classList.contains('active') || btn.classList.contains('selected')) return true;
            if (btn.parentElement && (btn.parentElement.classList.contains('is-checked') || btn.parentElement.classList.contains('active'))) return true;
            var parentSlide = btn.closest('.swiper-slide, li, div');
            if (parentSlide && (parentSlide.classList.contains('is-checked') || parentSlide.classList.contains('active'))) return true;
            return false;
        }

        // Helper to reliably trigger Samsung's custom web component event delegation
        function clickBtn(el) {
            if (!el) return;
            if (window.jQuery) {
                try { window.jQuery(el).trigger('click'); } catch(e){}
            }
            try {
                var rect = el.getBoundingClientRect();
                var x = rect.left + rect.width / 2;
                var y = rect.top + rect.height / 2;
                var opts = { bubbles: true, cancelable: true, view: window, clientX: x, clientY: y, screenX: x, screenY: y };
                ['pointerdown', 'mousedown', 'pointerup', 'mouseup', 'click'].forEach(function(evt) {
                    try { el.dispatchEvent(new MouseEvent(evt, opts)); } catch(e){}
                });
            } catch(e) {}
            try { el.click(); } catch(e2) {}
        }

        function getCleanText(el) {
            if (!el) return '';
            return (el.innerText || el.textContent || '').replace(/\\s+/g, ' ').trim();
        }

        function getButtonLabel(btn) {
            if (!btn) return '';
            var clone = btn.cloneNode(true);
            var blind = clone.querySelectorAll('.blind, .hidden');
            blind.forEach(function(b) { b.remove(); });
            return getCleanText(clone);
        }

        // 5. Color and Storage / Size Swatches
        var initialColors = Array.from(card.querySelectorAll('.option-selector-v2__color, [data-chiptype="color"]'));
        var initialSizes = Array.from(card.querySelectorAll('.option-selector-v2__size, [data-chiptype="other"], [data-chiptype="size"]'));

        if (initialColors.length > 0) {
            for (var cIdx = 0; cIdx < initialColors.length; cIdx++) {
                var freshCBtns = Array.from(card.querySelectorAll('.option-selector-v2__color, [data-chiptype="color"]'));
                var cBtn = freshCBtns[cIdx];
                if (!cBtn) continue;

                var colorName = (
                    cBtn.getAttribute('an-la') ? cBtn.getAttribute('an-la').replace(/^color:/i, '').trim() : ''
                ) || (
                    cBtn.getAttribute('title') ? cBtn.getAttribute('title').trim() : ''
                ) || (
                    cBtn.getAttribute('aria-label') ? cBtn.getAttribute('aria-label').trim() : ''
                );
                if (!colorName) {
                    colorName = getButtonLabel(cBtn);
                }
                if (!colorName) {
                    colorName = 'Color ' + (cIdx + 1);
                }

                // Click color if not already selected
                var buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
                if (!isBtnSelected(cBtn)) {
                    var prevSku = buyBtn ? buyBtn.getAttribute('data-modelcode') : null;
                    clickBtn(cBtn);
                    
                    var elapsed = 0;
                    var retried = false;
                    while (elapsed < 600) {
                        await new Promise(function(r) { setTimeout(r, 50); });
                        elapsed += 50;
                        freshCBtns = Array.from(card.querySelectorAll('.option-selector-v2__color, [data-chiptype="color"]'));
                        var liveCBtn = freshCBtns[cIdx];
                        buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
                        var currentSku = buyBtn ? buyBtn.getAttribute('data-modelcode') : null;
                        if (isBtnSelected(liveCBtn) && (prevSku === null || currentSku !== prevSku || currentSku === liveCBtn?.getAttribute('data-modelcode'))) {
                            break;
                        }
                        if (elapsed >= 250 && !retried) {
                            retried = true;
                            clickBtn(liveCBtn || cBtn);
                        }
                    }
                    await new Promise(function(r) { setTimeout(r, 120); });
                }

                // Query fresh storage options for this color
                var curStorageBtns = Array.from(card.querySelectorAll('.option-selector-v2__size, [data-chiptype="other"], [data-chiptype="size"]'));
                if (curStorageBtns.length > 0) {
                    for (var sIdx = 0; sIdx < curStorageBtns.length; sIdx++) {
                        var freshSBtns = Array.from(card.querySelectorAll('.option-selector-v2__size, [data-chiptype="other"], [data-chiptype="size"]'));
                        var sBtn = freshSBtns[sIdx];
                        if (!sBtn) continue;

                        var isStorageDisabled = sBtn.hasAttribute('disabled') || sBtn.classList.contains('disabled');
                        var expectedCode = sBtn.getAttribute('data-modelcode');
                        var storageName = getButtonLabel(sBtn) || ('Option ' + (sIdx + 1));

                        buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');

                        if (!isStorageDisabled && !isBtnSelected(sBtn)) {
                            var prevSku = buyBtn ? buyBtn.getAttribute('data-modelcode') : null;
                            clickBtn(sBtn);

                            var elapsed = 0;
                            var retried = false;
                            while (elapsed < 600) {
                                await new Promise(function(r) { setTimeout(r, 50); });
                                elapsed += 50;
                                freshSBtns = Array.from(card.querySelectorAll('.option-selector-v2__size, [data-chiptype="other"], [data-chiptype="size"]'));
                                var liveSBtn = freshSBtns[sIdx];
                                buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
                                var currentSku = buyBtn ? buyBtn.getAttribute('data-modelcode') : null;
                                if (isBtnSelected(liveSBtn) && (prevSku === null || currentSku !== prevSku || currentSku === liveSBtn?.getAttribute('data-modelcode'))) {
                                    break;
                                }
                                if (elapsed >= 250 && !retried) {
                                    retried = true;
                                    clickBtn(liveSBtn || sBtn);
                                }
                            }
                            await new Promise(function(r) { setTimeout(r, 100); });
                        }

                        // Prices
                        buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
                        var priceWrap = card.querySelector('.pd21-product-card__price-main, .price-ux__wrap, .price-ux__price-current, .pd21-product-card__price, [class*="buying-price"]');
                        var priceRaw = priceWrap ? priceWrap.innerText.trim().replace(/\\s+/g, ' ') : '';
                        var wasEl = card.querySelector('.pd21-product-card__was-price, .price-ux__was-price, .price-ux__price-original, [class*="was-price"], [class*="origin-price"]');
                        var wasRaw = wasEl ? wasEl.innerText.trim() : '';

                        // Stock status
                        var notifyBtn = card.querySelector('.cta--notify, [class*="notify"]');
                        var stock = 'In Stock';
                        if (isStorageDisabled) {
                            stock = 'Out of Stock / Unavailable';
                        } else if (notifyBtn || (buyBtn && /notify/i.test(buyBtn.innerText))) {
                            stock = 'Notify Me';
                        } else if (buyBtn && (buyBtn.hasAttribute('disabled') || buyBtn.classList.contains('disabled'))) {
                            stock = 'Out of Stock';
                        }

                        // Model code / SKU - guaranteed by buyBtn active modelcode or expectedCode
                        var sku = (buyBtn ? buyBtn.getAttribute('data-modelcode') : '') ||
                                  expectedCode ||
                                  cBtn.getAttribute('data-modelcode') || '';

                        var learnMore = card.querySelector('a.js-pfv2-learn-more, a.pd21-product-card__name');
                        var itemUrl = (learnMore ? learnMore.getAttribute('href') : '') || (buyBtn ? buyBtn.getAttribute('href') : '');

                        results.push({
                            name: baseName,
                            color: colorName,
                            storage: storageName,
                            sku: sku,
                            price_raw: priceRaw,
                            was_raw: wasRaw,
                            badge: badgeStr,
                            offers: offersStr,
                            rating: ratingScore,
                            rating_count: exactCount,
                            stock: stock,
                            url: itemUrl
                        });
                    }
                } else {
                    // Color only, no storage chips
                    buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
                    var priceWrap = card.querySelector('.pd21-product-card__price-main, .price-ux__wrap, .price-ux__price-current, .pd21-product-card__price, [class*="buying-price"]');
                    var priceRaw = priceWrap ? priceWrap.innerText.trim().replace(/\\s+/g, ' ') : '';
                    var wasEl = card.querySelector('.pd21-product-card__was-price, .price-ux__was-price, .price-ux__price-original, [class*="was-price"]');
                    var wasRaw = wasEl ? wasEl.innerText.trim() : '';

                    var notifyBtn = card.querySelector('.cta--notify, [class*="notify"]');
                    var stock = 'In Stock';
                    if (notifyBtn || (buyBtn && /notify/i.test(buyBtn.innerText))) {
                        stock = 'Notify Me';
                    } else if (buyBtn && (buyBtn.hasAttribute('disabled') || buyBtn.classList.contains('disabled'))) {
                        stock = 'Out of Stock';
                    }

                    var sku = (buyBtn ? buyBtn.getAttribute('data-modelcode') : '') ||
                              cBtn.getAttribute('data-modelcode') ||
                              (nameEl ? nameEl.getAttribute('data-modelcode') : '') || '';

                    var learnMore = card.querySelector('a.js-pfv2-learn-more, a.pd21-product-card__name');
                    var itemUrl = (learnMore ? learnMore.getAttribute('href') : '') || (buyBtn ? buyBtn.getAttribute('href') : '');

                    results.push({
                        name: baseName,
                        color: colorName,
                        storage: 'Default',
                        sku: sku,
                        price_raw: priceRaw,
                        was_raw: wasRaw,
                        badge: badgeStr,
                        offers: offersStr,
                        rating: ratingScore,
                        rating_count: exactCount,
                        stock: stock,
                        url: itemUrl
                    });
                }
            }
        } else if (initialSizes.length > 0) {
            // Storage/size chips without color chips (e.g. TVs or Monitors)
            for (var sIdx = 0; sIdx < initialSizes.length; sIdx++) {
                var freshSBtns = Array.from(card.querySelectorAll('.option-selector-v2__size, [data-chiptype="other"], [data-chiptype="size"]'));
                var sBtn = freshSBtns[sIdx];
                if (!sBtn) continue;

                var isStorageDisabled = sBtn.hasAttribute('disabled') || sBtn.classList.contains('disabled');
                var expectedCode = sBtn.getAttribute('data-modelcode');
                var storageName = getButtonLabel(sBtn) || ('Option ' + (sIdx + 1));

                var buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');

                if (!isStorageDisabled && !isBtnSelected(sBtn)) {
                    var prevSku = buyBtn ? buyBtn.getAttribute('data-modelcode') : null;
                    clickBtn(sBtn);

                    var elapsed = 0;
                    var retried = false;
                    while (elapsed < 600) {
                        await new Promise(function(r) { setTimeout(r, 50); });
                        elapsed += 50;
                        freshSBtns = Array.from(card.querySelectorAll('.option-selector-v2__size, [data-chiptype="other"], [data-chiptype="size"]'));
                        var liveSBtn = freshSBtns[sIdx];
                        buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
                        var currentSku = buyBtn ? buyBtn.getAttribute('data-modelcode') : null;
                        if (isBtnSelected(liveSBtn) && (prevSku === null || currentSku !== prevSku || currentSku === liveSBtn?.getAttribute('data-modelcode'))) {
                            break;
                        }
                        if (elapsed >= 250 && !retried) {
                            retried = true;
                            clickBtn(liveSBtn || sBtn);
                        }
                    }
                    await new Promise(function(r) { setTimeout(r, 100); });
                }

                buyBtn = card.querySelector('a.pd21-product-card__image-cta, a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
                var priceWrap = card.querySelector('.pd21-product-card__price-main, .price-ux__wrap, .price-ux__price-current, .pd21-product-card__price, [class*="buying-price"]');
                var priceRaw = priceWrap ? priceWrap.innerText.trim().replace(/\\s+/g, ' ') : '';
                var wasEl = card.querySelector('.pd21-product-card__was-price, .price-ux__was-price, .price-ux__price-original, [class*="was-price"]');
                var wasRaw = wasEl ? wasEl.innerText.trim() : '';

                var notifyBtn = card.querySelector('.cta--notify, [class*="notify"]');
                var stock = 'In Stock';
                if (isStorageDisabled) {
                    stock = 'Out of Stock / Unavailable';
                } else if (notifyBtn || (buyBtn && /notify/i.test(buyBtn.innerText))) {
                    stock = 'Notify Me';
                }

                var sku = (buyBtn ? buyBtn.getAttribute('data-modelcode') : '') ||
                          expectedCode ||
                          (nameEl ? nameEl.getAttribute('data-modelcode') : '') || '';

                var learnMore = card.querySelector('a.js-pfv2-learn-more, a.pd21-product-card__name');
                var itemUrl = (learnMore ? learnMore.getAttribute('href') : '') || (buyBtn ? buyBtn.getAttribute('href') : '');

                results.push({
                    name: baseName,
                    color: 'Default',
                    storage: storageName,
                    sku: sku,
                    price_raw: priceRaw,
                    was_raw: wasRaw,
                    badge: badgeStr,
                    offers: offersStr,
                    rating: ratingScore,
                    rating_count: exactCount,
                    stock: stock,
                    url: itemUrl
                });
            }
        } else {
            // Single variant product (no swatches)
            var priceWrap = card.querySelector('.pd21-product-card__price-main, .price-ux__wrap, .price-ux__price-current, .pd21-product-card__price, [class*="buying-price"]');
            var priceRaw = priceWrap ? priceWrap.innerText.trim().replace(/\\s+/g, ' ') : '';
            var wasEl = card.querySelector('.pd21-product-card__was-price, .price-ux__was-price, .price-ux__price-original, [class*="was-price"]');
            var wasRaw = wasEl ? wasEl.innerText.trim() : '';

            var buyBtn = card.querySelector('a.js-pfv2-buy-now, [class*="buy-now"], .cta--primary');
            var notifyBtn = card.querySelector('.cta--notify, [class*="notify"]');
            var stock = 'In Stock';
            if (notifyBtn || (buyBtn && /notify/i.test(buyBtn.innerText))) {
                stock = 'Notify Me';
            } else if (buyBtn && (buyBtn.hasAttribute('disabled') || buyBtn.classList.contains('disabled'))) {
                stock = 'Out of Stock';
            }

            var sku = (buyBtn ? buyBtn.getAttribute('data-modelcode') : '') ||
                      (nameEl ? nameEl.getAttribute('data-modelcode') : '') || '';

            var learnMore = card.querySelector('a.js-pfv2-learn-more, a.pd21-product-card__name');
            var itemUrl = (learnMore ? learnMore.getAttribute('href') : '') || (buyBtn ? buyBtn.getAttribute('href') : '');

            results.push({
                name: baseName,
                color: 'Default',
                storage: 'Default',
                sku: sku,
                price_raw: priceRaw,
                was_raw: wasRaw,
                badge: badgeStr,
                offers: offersStr,
                rating: ratingScore,
                rating_count: exactCount,
                stock: stock,
                url: itemUrl
            });
        }

        callback(results);
    })();
    """

    num_cards = driver.execute_script("""
        return document.querySelectorAll('.pd21-product-card__item.js-pfv2-product-card, .pd21-product-card__item, .pd03-product-card').length;
    """) or 0

    if max_cards and max_cards > 0:
        total_to_process = min(num_cards, max_cards)
    else:
        total_to_process = num_cards

    print(f"[Catalog Scraper] Found {num_cards} catalog cards. Extracting variants card by card...")

    driver.set_script_timeout(35)
    raw_results = []

    for c_idx in range(total_to_process):
        try:
            card_items = driver.execute_async_script(js_card_extractor, c_idx) or []
            if card_items:
                c_name = card_items[0].get("name", f"Card {c_idx + 1}")
                print(f"  [{c_idx + 1}/{total_to_process}] {c_name} -> {len(card_items)} variant(s)")
                raw_results.extend(card_items)
            else:
                print(f"  [{c_idx + 1}/{total_to_process}] Card {c_idx + 1} -> 0 variants")
        except Exception as e:
            print(f"  [{c_idx + 1}/{total_to_process}] Warning: Failed extracting card {c_idx + 1}: {e}")

    print(f"[Catalog Scraper] Completed! Extracted {len(raw_results)} total variant records.")

    items = []
    seen = set()
    for item in raw_results:
        name = item.get("name", "Unknown Product").strip()
        color = item.get("color", "Default").strip()
        storage = item.get("storage", "Default").strip()
        sku = item.get("sku", "").strip().upper()
        if not sku:
            sku = f"SAMS-{abs(hash(name + color + storage)) % 10000000}"

        key = (name, color, storage, sku)
        if key in seen:
            continue
        seen.add(key)

        price_raw = item.get("price_raw", "")
        was_raw = item.get("was_raw", "")

        curr_price = "Not Available"
        m_or = re.search(r'or\s+\$([0-9,]+\.?[0-9]*)', price_raw, re.IGNORECASE)
        if m_or:
            curr_price = f"${float(m_or.group(1).replace(',', '')):,.2f}"
        else:
            m_p = re.search(r'\$([0-9,]+\.[0-9]{2})', price_raw)
            if m_p:
                curr_price = f"${float(m_p.group(1).replace(',', '')):,.2f}"

        orig_price = "Not Available"
        m_was = re.search(r'\$([0-9,]+\.?[0-9]*)', was_raw)
        if m_was:
            orig_price = f"${float(m_was.group(1).replace(',', '')):,.2f}"

        fin_opt = "Not Applicable"
        m_fin = re.search(r'((?:From\s+)?\$[0-9,]+(?:\.[0-9]{2})?/mo(?:\s+for\s+\d+\s+mos)?)', price_raw, re.IGNORECASE)
        if m_fin:
            fin_opt = m_fin.group(1).strip()

        p_url = item.get("url", "")
        if p_url and not p_url.startswith("http"):
            p_url = "https://www.samsung.com" + p_url

        items.append({
            'Model Name': name,
            'Color': color,
            'Storage': storage,
            'SKU Code': sku,
            'Product Price': curr_price,
            'Original Price': orig_price,
            'Financing Option': fin_opt,
            'Badge': item.get("badge", "None"),
            'Offers': item.get("offers", "None"),
            'Product Rating': item.get("rating") if item.get("rating") is not None else 'Not Rated',
            'Number of Ratings': item.get("rating_count", 0),
            'Stock Status': item.get("stock", "In Stock"),
            'Product URL': p_url
        })

    return items


## **6. Category Scraper Runner Function**

The function below scrapes a category directly from its catalog cards, ensures its designated folder exists, and writes the output `.csv` file.

In [51]:
def scrape_category(driver, category_name, category_config, max_products=None):
    """
    Scrapes products for a specified category directly from its catalog page cards,
    extracting all Color and Storage variants, pricing, financing, badges, offers, exact ratings, and stock status.
    Set max_products=N to limit to N cards (useful for testing).
    """
    url = category_config["url"]
    folder = category_config["folder"]
    filename = category_config["filename"]
    
    # Ensure output folder exists
    os.makedirs(folder, exist_ok=True)
    csv_path = os.path.join(folder, filename)
    
    print("\n" + "="*65)
    print(f"Category:    {category_name}")
    print(f"Target URL:  {url}")
    print(f"Destination: {csv_path}")
    print("="*65)
    
    # Extract variant records directly from catalog cards
    items = scrape_catalog_page_variants(driver, category_name, url, max_cards=max_products)
    
    # Save to CSV inside the designated category folder
    df = pd.DataFrame(items)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"Successfully saved {len(df)} variant records to: {csv_path}")
    return df

## **7. Quick Test on a Single Category (e.g. Smartphones)**

Run this cell to test scraping the first 2-3 product cards from a single category to verify the variant matrix, color swatches, storage choices, badges, offers, exact ratings, and `.csv` output.

In [52]:
# Test run on Smartphones (limited to 2 cards to test full variant extraction)
driver = init_webdriver(headless=False)
try:
    test_df = scrape_category(
        driver,
        category_name="Smartphones",
        category_config=CATEGORIES["Smartphones"],
        max_products=2
    )
    display(test_df)
finally:
    driver.quit()


Category:    Smartphones
Target URL:  https://www.samsung.com/ca/smartphones/all-smartphones/
Destination: scraped_data\Smartphones\samsung_smartphones.csv

[Catalog Scraper] Loading category: Smartphones -> https://www.samsung.com/ca/smartphones/all-smartphones/
[Catalog Scraper] Found 37 catalog cards. Extracting variants card by card...
  [1/2] Galaxy S26 FE -> 6 variant(s)
  [2/2] Galaxy Z Fold8 (Samsung.com only) -> 2 variant(s)
[Catalog Scraper] Completed! Extracted 8 total variant records.
Successfully saved 8 variant records to: scraped_data\Smartphones\samsung_smartphones.csv


,Model Name,Color,Storage,SKU Code,Product Price,Original Price,Financing Option,Badge,Offers,Product Rating,Number of Ratings,Stock Status,Product URL
0,Galaxy S26 FE,pistachio,256 GB,SM-S741WLGEXAC,"$1,189.99",Not Available,From $49.58/mo for 24 mos,New,Save up to $490 with an eligible trade-in,Not Rated,0,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...
1,Galaxy S26 FE,pistachio,128 GB,SM-S741WLGAXAC,"$1,049.99",Not Available,From $43.75/mo for 24 mos,New,Save up to $490 with an eligible trade-in,Not Rated,0,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...
2,Galaxy S26 FE,blueberry,256 GB,SM-S741WZVEXAC,"$1,189.99",Not Available,From $49.58/mo for 24 mos,New,Save up to $490 with an eligible trade-in,Not Rated,0,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...
3,Galaxy S26 FE,blueberry,128 GB,SM-S741WZVAXAC,"$1,049.99",Not Available,From $43.75/mo for 24 mos,New,Save up to $490 with an eligible trade-in,Not Rated,0,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...
4,Galaxy S26 FE,graphite,256 GB,SM-S741WZKEXAC,"$1,189.99",Not Available,From $49.58/mo for 24 mos,New,Save up to $490 with an eligible trade-in,Not Rated,0,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...
5,Galaxy S26 FE,graphite,128 GB,SM-S741WZKAXAC,"$1,049.99",Not Available,From $43.75/mo for 24 mos,New,Save up to $490 with an eligible trade-in,Not Rated,0,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...
6,Galaxy Z Fold8 (Samsung.com only),pistachio,512 GB,SM-F971WZGEXAC,"$2,679.99",Not Available,From $111.67/mo for 24 mos,New,None,4.9,4603,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...
7,Galaxy Z Fold8 (Samsung.com only),pistachio,256 GB,SM-F971WZGAXAC,"$2,399.99",Not Available,From $100.00/mo for 24 mos,New,None,4.9,4603,In Stock,https://www.samsung.com/ca/smartphones/galaxy-...


## **8. Full Execution: Scrape ALL Categories into Separate Folders & CSVs**

Execute the cell below to scrape all 11 product categories directly from their catalog pages.
- Each category creates its own folder inside `scraped_data/`.
- Each category saves its own `.csv` file inside that folder.
- Leave `MAX_PER_CATEGORY = None` to scrape all product cards in every category.

In [53]:
# Set MAX_PER_CATEGORY (None for all product cards)
MAX_PER_CATEGORY = None

driver = init_webdriver(headless=True)
results_summary = {}

try:
    for cat_name, cat_config in CATEGORIES.items():
        df = scrape_category(
            driver,
            category_name=cat_name,
            category_config=cat_config,
            max_products=MAX_PER_CATEGORY
        )
        results_summary[cat_name] = {
            "Total Variants Scraped": len(df),
            "Output Folder": cat_config["folder"],
            "CSV File": cat_config["filename"]
        }
finally:
    driver.quit()

# Display scraping run summary
summary_df = pd.DataFrame.from_dict(results_summary, orient="index")
print("\n" + "="*65)
print("ALL CATEGORIES SCRAPING COMPLETE - SUMMARY")
print("="*65)
display(summary_df)


Category:    Smartphones
Target URL:  https://www.samsung.com/ca/smartphones/all-smartphones/
Destination: scraped_data\Smartphones\samsung_smartphones.csv

[Catalog Scraper] Loading category: Smartphones -> https://www.samsung.com/ca/smartphones/all-smartphones/
[Catalog Scraper] Found 37 catalog cards. Extracting variants card by card...
  [1/37] Galaxy S26 FE -> 6 variant(s)
  [2/37] Galaxy Z Fold8 (Samsung.com only) -> 2 variant(s)
  [3/37] Galaxy Z Fold8 Ultra (Samsung.com only) -> 1 variant(s)
  [4/37] Card 4 -> 0 variants
  [5/37] Galaxy Z Flip8 (Samsung.com only) -> 2 variant(s)
  [6/37] Galaxy Z Fold8 -> 9 variant(s)
  [7/37] Galaxy Z Fold8 Ultra -> 9 variant(s)
  [8/37] Galaxy Z Flip8 -> 6 variant(s)
  [9/37] Galaxy S26 Ultra (Samsung.com only) -> 6 variant(s)
  [10/37] Galaxy S26 Ultra -> 9 variant(s)
  [11/37] Galaxy Z Flip7 -> 1 variant(s)
  [12/37] Card 12 -> 0 variants
  [13/37] Galaxy S26+ (Samsung.com only) -> 4 variant(s)
  [14/37] Galaxy S26 (Samsung.com only) -> 4 

,Total Variants Scraped,Output Folder,CSV File
Smartphones,118,scraped_data\Smartphones,samsung_smartphones.csv
Tablets,35,scraped_data\Tablets,samsung_tablets.csv
Watches,25,scraped_data\Watches,samsung_watches.csv
Audio_and_Buds,7,scraped_data\Audio_and_Buds,samsung_audio_and_buds.csv
TVs,223,scraped_data\TVs,samsung_tvs.csv
Sound_Devices,32,scraped_data\Sound_Devices,samsung_sound_devices.csv
Refrigerators,156,scraped_data\Refrigerators,samsung_refrigerators.csv
Laundry,73,scraped_data\Laundry,samsung_laundry.csv
Cooking_Appliances,94,scraped_data\Cooking_Appliances,samsung_cooking_appliances.csv
Monitors,59,scraped_data\Monitors,samsung_monitors.csv


---
## **9. Product Intelligence, SQLite Database & Price Tracking**

The data scraped across categories is now automatically modeled into structured schemas and persisted into an **SQLite database (`data/samsung_tracker.db`)**.

Run the cell below to load the complete product catalog directly from SQLite into pandas with computed derived metrics (e.g. **Cost Per Screen Inch** for TVs, **Cost Per GB** for mobile devices).

In [54]:
from src.db import SamsungDB
import pandas as pd

# Initialize database connection
db = SamsungDB()
df = db.get_latest_products_df()

print(f"Loaded {len(df)} products from SQLite database across categories: {df['category'].unique().tolist()}")
display(df.head(10))

Loaded 80 products from SQLite database across categories: ['Smartphones', 'TVs', 'Tablets']


,sku,name,category,product_url,image_url,specs_json,current_price,original_price,discount_amount,discount_percentage,price_formatted,original_price_formatted,financing_option,rating,rating_count,availability,scraped_at,screen_size_inch,price_per_inch
0,SM-A156WZKAXAC,Galaxy A15 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},NaN,NaN,NaN,NaN,Not Available,Not Available,Choose convenient Instalment plans provided by...,3.7,101,In Stock,2026-09-08T23:00:10.077357+00:00,NaN,NaN
1,SM-A166WZKAXAC,Galaxy A16 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},NaN,NaN,NaN,NaN,Not Available,Not Available,Choose convenient Instalment plans provided by...,4.3,5701,In Stock,2026-09-08T23:00:10.077357+00:00,NaN,NaN
2,A166WZKAXAC,Galaxy A16 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},NaN,NaN,NaN,NaN,Not Available,Not Available,Not Applicable,4.3,5701,In Stock,2026-09-08T23:31:13.519828+00:00,NaN,NaN
3,SM-A176WZKAXAC,Galaxy A17 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},339.99,339.99,0.0,0.0,$339.99,Not Available,Choose convenient Instalment plans provided by...,3.6,218,In Stock,2026-09-08T23:00:10.076828+00:00,NaN,NaN
4,A176WZKAXAC,Galaxy A17 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},339.99,339.99,0.0,0.0,$339.99,Not Available,From $14.17/mo for 24 mos,3.6,213,In Stock,2026-09-08T23:31:13.518816+00:00,NaN,NaN
5,SM-A376WZAAXAC,Galaxy A37 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},449.99,449.99,0.0,0.0,$449.99,Not Available,Choose convenient Instalment plans provided by...,4.7,970,In Stock,2026-09-08T23:00:10.076828+00:00,NaN,NaN
6,A376WZAAXAC,Galaxy A37 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},449.99,449.99,0.0,0.0,$449.99,Not Available,From $18.75/mo for 24 mos,4.7,956,In Stock,2026-09-08T23:31:13.518308+00:00,NaN,NaN
7,SM-A566WZKAXAC,Galaxy A56 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},NaN,NaN,NaN,NaN,Not Available,Not Available,Choose convenient Instalment plans provided by...,4.5,55,In Stock,2026-09-08T23:00:10.077357+00:00,NaN,NaN
8,A566WZKAXAC,Galaxy A56 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},NaN,NaN,NaN,NaN,Not Available,Not Available,Not Applicable,4.5,55,In Stock,2026-09-08T23:31:13.519828+00:00,NaN,NaN
9,SM-A576WDBAXAC,Galaxy A57 5G,Smartphones,https://www.samsung.com/ca/smartphones/galaxy-...,,{},549.99,549.99,0.0,0.0,$549.99,Not Available,Choose convenient Instalment plans provided by...,4.7,1876,In Stock,2026-09-08T23:00:10.076828+00:00,NaN,NaN


## **10. Fast API & Schema.org JSON-LD Harvester**

Instead of relying solely on heavy Selenium browser instances, you can use `SamsungApiClient` to fetch structured product specifications, pricing, and ratings instantaneously via Schema.org JSON-LD and meta tags.

In [55]:
from src.api_client import SamsungApiClient

client = SamsungApiClient()
sample_url = "https://www.samsung.com/ca/tvs/oled-tv/s90c-55-inch-oled-4k-smart-tv-qn55s90cafxzc/"
product = client.fetch_product_from_url(sample_url, category="TVs")

print("SKU:       ", product.sku)
print("Name:      ", product.name)
print("Price:     ", product.price_formatted)
print("Specs:     ", product.specs)
print("Rating:    ", product.rating, f"({product.rating_count} reviews)")

SKU:        QN55S90CAFXZC
Name:       Samsung TVs | Shop OLED, Neo QLED, 4K & More
Price:      Not Available
Specs:      {'display_tech': 'Neo QLED', 'resolution': '4K UHD (3840 x 2160)', 'refresh_rate': '60Hz', 'screen_size_inch': 55.0}
Rating:     None (0 reviews)


## **11. Price-Drop & Deal Alert Engine**

The `PriceAlertEngine` automatically analyzes historical price records in SQLite to detect:
1. **Price Drops** (current price < previous scrape)
2. **All-Time Lows** (cheapest price recorded for that SKU)
3. **Steep Discounts** (promotions with >= 15% off regular price)

In [56]:
from src.alerts import PriceAlertEngine

engine = PriceAlertEngine(db, discount_threshold=15.0)
alerts = engine.check_alerts()

print(f"Price Drops Detected: {len(alerts['price_drops'])}")
print(f"All-Time Lows:        {len(alerts['all_time_lows'])}")
print(f"Steep Discounts:      {len(alerts['steep_discounts'])}")

if alerts["price_drops"]:
    print("\n--- Active Price Drops ---")
    display(pd.DataFrame(alerts["price_drops"]))

if alerts["steep_discounts"]:
    print("\n--- Steep Discounts ---")
    display(pd.DataFrame(alerts["steep_discounts"])[["name", "current_price", "original_price", "discount_amount", "discount_percentage"]].head(5))

Price Drops Detected: 1
All-Time Lows:        38
Steep Discounts:      0

--- Active Price Drops ---


,sku,name,category,old_price,new_price,drop_amount,drop_percentage,timestamp
0,GALAXY-S26,Galaxy S26,Smartphones,1809.99,1529.99,280.0,15.5,2026-09-08T23:31:13.518308+00:00


## **12. Value-for-Money & Price Trend Visualizations**

Analyze market positioning, cost efficiency, and historical price curves interactively using Plotly.

In [ ]:
# import plotly.express as px

# # 1. TV Screen Size vs. Price (Bubble size = Cost per Inch)
# tv_df = df[df["category"] == "TVs"].dropna(subset=["screen_size_inch", "current_price"]).copy()
# if not tv_df.empty:
#     fig_tv = px.scatter(
#         tv_df,
#         x="screen_size_inch",
#         y="current_price",
#         size="price_per_inch",
#         color="display_tech" if "display_tech" in tv_df.columns else None,
#         hover_name="name",
#         title="Samsung TVs: Screen Size vs. Price (Bubble Size = Cost per Inch)",
#         labels={"screen_size_inch": "Screen Size (inches)", "current_price": "Price (CAD)"}
#     )
#     fig_tv.update_layout(template="plotly_dark")
#     fig_tv.show()

import plotly.express as px
import plotly.io as pio

# Ensure plots render cleanly in VS Code notebooks
pio.renderers.default = "vscode"

# 1. TV Screen Size vs. Price (Bubble size = Cost per Inch)
tv_df = df[df["category"] == "TVs"].dropna(subset=["screen_size_inch", "current_price"]).copy()
if not tv_df.empty:
    fig_tv = px.scatter(
        tv_df,
        x="screen_size_inch",
        y="current_price",
        size="price_per_inch",
        color="display_tech" if "display_tech" in tv_df.columns else None,
        hover_name="name",
        title="Samsung TVs: Screen Size vs. Price (Bubble Size = Cost per Inch)",
        labels={"screen_size_inch": "Screen Size (inches)", "current_price": "Price (CAD)"}
    )
    fig_tv.update_layout(template="plotly_dark")
    fig_tv.show()


# 2. Historical Price Movement Trend for a Specific Model
sample_sku = "QN55S90CAFXZC"
history_df = db.get_price_history_df(sample_sku)
if not history_df.empty:
    fig_hist = px.line(
        history_df,
        x="scraped_at",
        y="current_price",
        markers=True,
        title=f"Historical Price Trend for SKU: {sample_sku}",
        labels={"scraped_at": "Timestamp", "current_price": "Price (CAD)"}
    )
    fig_hist.update_layout(template="plotly_dark")
    fig_hist.show()

## **13. Interactive Web Dashboard**

You can also run the complete **Streamlit Analytics Dashboard** (`app.py`) for a full web UI with deal filters, side-by-side comparison, and real-time alerts.

To launch the dashboard:
- Run in your terminal: `python -m streamlit run app.py`
- Or double-click `run_dashboard.bat` in the project root.